# Tarea 2 — Respuesta al impulso  

#### Integrantes: Dafne Angulo - Tomas Araya - Antonia Jimenez

#### Profesores: Dr. Victor Poblete - Felipe Figueroa

#### ACUS099: Procesamiento digital de señales.

#### Jueves 30 de abril

# 1) Introducción 

Esta página web se diseño para mostrar el procedimiento y registrar la respuesta de impulso para 3 recintos dentro de la UACH. Se utilizaron dos grabaciones hechas en la cámara anecoica: Un trabalenguas dicho por Tomás Araya y un trompe interpretrado por Antonia Jimenez, siendo así estas las referencias para evidenciar como cada recinto modifica los registros originales mediante el proceso de convolución.

# 2) Objetivos

## 2.1) Objetivo general

Analizar la percepción de las señales acústicas en los distintos espacios medidos.

## 2.2) Objetivos especificos

a. Generar señales de referencia. 

b. Realizar mediciones mediante respuesta al impulso según ISO 3382.

c. Aplicar convolución a las referencias con las características de cada espacio.

# 3) Marco teórico

La convolución es el proceso mediante el cual una señal de entrada se ve modificada al interactuar con otra señal, generando como resultado una nueva señal que incorpora las características de ambas. En este caso, la señal de entrada corresponde a las grabaciones realizadas en la cámara anecoica, las cuales se combinan con las respuestas de las demás salas. 

En este contexto, la convolución se expresa mediante la siguiente relación:

                                               y(t)=x(t)∗h(t)                                              

donde:

x (t) : Corresponde a la señal de entrada, que en este caso son las grabaciones realizadas en la cámara reverberante.  
h (t) : Representa la respuesta al impulso de las demas salas.            
y (t) : Es la señal resultante, que contiene el sonido original modificado por el comportamiento acústico de cada recinto.

# 4) Procedimiento

Para realizar las mediciones de respuestas de impulso, se utilizaron los siguientes equipos y programas: 

- 1 Micrófono Behringer 8000
- 1 Cable XLR de 6 metros
- 1 Atril para micrófono 
- 1 Claqueta
- 1 Trompe
- 1 Interfaz ESI U22 XT
- 1 Interfaz PreSonus 26c
- Software Reaper

Los recintos donde se analizó la respuesta al impulso fueron: 
- La cámara anecoica del laboratorio de acústica
- La cámara reverberante 
- El hall del edificio 9.000
- Laboratorio LATE 2 del edificio 14.000
Todos las mediciones fueron realizadas en dias distintos.

En la cámara anecoica realizamos dos grabaciones: la primera de un trabalenguas, interpretado por Tomás Araya, y la segunda de un trompe, tocado por Antonia Jiménez, como se menciono anteriormente.

Luego, para las demás salas, realizamos seis grabaciones, con tres posiciones de micrófono y dos posiciones de fuente, con tres claqueteos por posición, dando así 18 mediciones por recinto.

# Desarrollo

El codigo utilizar para convolucionar la voz y el trompe son idénticos, y solo se cambio el archivo de audio utilizado.

Por conveniencia, se muestra a continuación el codigo utilizado para convolucionar la voz, pero se mostrará el resultado para ambos casos.

Además, para el ejemplo del código solo se muestra el proceso de la cámara reverberante, pero se mostrarán los gráficos y audios correspondientes para el Hall del edificio 9.000 y el laboratorio LATE 2.

1) Se importan las librerías a utilizar

In [15]:
# 1. Importar librerías

from pathlib import Path #Libreria para gestionar la ruta de los archivos
import numpy as np #Libreria para el manejo de arrays y operaciones matematicas
import matplotlib.pyplot as plt #Libreria para graficar
import soundfile as sf #Libreria para leer y escribir archivos de audio
import librosa #Libreria para el procesamiento de audio
from scipy.signal import fftconvolve #Libreria para realizar la convolucion rapida
from IPython.display import Audio, Image, display #Libreria para reproducir audio en Jupyter Notebook
from PIL import Image #Libreria para el manejo de imagenes

2) Se verifica la ruta de los archivos a utilizar:

In [16]:
# 2. Definir rutas de los archivos de audio
# 2.1. Rutas de audio para la voz y el instrumento
ruta_voz = Path("../assets/audio/Grabaciones_propias/Voz_e_Instrumento/Grabacion de voz ANECOICA.wav") #Color verde

# 2.2. Rutas de para la cámara reverberante
ruta_rev = Path("../assets/audio/Grabaciones_propias/Respuestas_de_impulso/Reverberante/Respuesta de impulso reverberante.wav") #REVERBERANTE, color rojo

# 2.3, Verificar que existan

print("Ruta voz:", ruta_voz)
print("Ruta IR Camara Reverberante:", ruta_rev)

assert ruta_voz.exists(), f"No existe el archivo: {ruta_voz}" #Verificar que el archivo de voz exista
assert ruta_rev.exists(), f"No existe el archivo: {ruta_rev}" #Verificar que el archivo de la cámara reverberante exista

Ruta voz: ..\assets\audio\Grabaciones_propias\Voz_e_Instrumento\Grabacion de voz ANECOICA.wav
Ruta IR Camara Reverberante: ..\assets\audio\Grabaciones_propias\Respuestas_de_impulso\Reverberante\Respuesta de impulso reverberante.wav


3) Se cargan y se ajustan los archivos para que tengan la misma frecuencia de muestreo antes de realizar la convolución. En este caso, se muestra el proceso llevado a cabo para la cámara reverberante, pero es el mismo para el hall del edificio 9.000 y para el laboratorio LATE 2:

In [17]:
# 3. Cargar archivos y ajustar señales para la reverberante

#Voz
# Cargar archivos
voz, fs_voz = sf.read(ruta_voz) #sf.read devuelve un array con la señal de audio y la frecuencia de muestreo
ir_rev, fs_ir_rev = sf.read(ruta_rev)

# ---------------------------------------------------------
# Ajuste de canales (mono)
# ---------------------------------------------------------
if voz.ndim > 1: #Si la voz tiene más de un canal, promediar para obtener mono
    voz = voz.mean(axis=1)

if ir_rev.ndim > 1:
    ir_rev = ir_rev.mean(axis=1)

# ---------------------------------------------------------
# Ajuste de frecuencia de muestreo
# ---------------------------------------------------------
if fs_voz != fs_ir_rev: 
    # Si las frecuencias de muestreo no coinciden, se remuestrea la voz para que coincida con 
    # la frecuencia de muestreo de la respuesta de impulso reverberante
    print("Las frecuencias no coinciden. Remuestreando voz...")
    voz = librosa.resample(
        voz.astype(float),
        orig_sr=fs_voz,
        target_sr=fs_ir_rev
    )
    fs_voz = fs_ir_rev #Actualizar la frecuencia de muestreo de la voz para que coincida con la de la respuesta de impulso reverberante

# ---------------------------------------------------------
# Verificación final
# ---------------------------------------------------------
print("Frecuencia de muestreo voz:", fs_voz)
print("Frecuencia de muestreo impulso reverberante:", fs_ir_rev)

assert fs_voz == fs_ir_rev, "Las frecuencias de muestreo de la voz con la reverberante deben coincidir" #Verificar que las frecuencias de muestreo coincidan después del ajuste

Frecuencia de muestreo voz: 44100
Frecuencia de muestreo impulso reverberante: 44100


4) Convolucionar las señales:

In [18]:
# 4. Convolucionar
# 4.1. Convolución reverberante

yrev = fftconvolve(voz, ir_rev, mode="full") 

# La convolución con la respuesta de impulso reverberante se realiza utilizando la función fftconvolve de scipy.signal, 
# que es una implementación eficiente de la convolución utilizando la Transformada Rápida de Fourier (FFT).
# El resultado de la convolución se almacena en la variable yrev, que representa la señal de audio resultante de aplicar
# la reverberación a la voz original.

# Normalizar para evitar clipping
if np.max(np.abs(yrev)) > 0:
    yrev = yrev / np.max(np.abs(yrev)) 
    
# Normalizar la señal resultante de la convolución para evitar que los valores 
# superen el rango permitido para archivos de audio, lo que podría causar distorsión (clipping) al reproducir o guardar el archivo.
 

Ya con esto, se pueden realizar los gráficos correspondientes para cada recinto, que se apreciaran en sus respectivos apartados.

## Recinto 0: Camara Anecoica 

Como se menciono anteriormente, en este recinto se realizaron las grabaciones de un trabalenguas y el sonido de un trompe, 
las cuales, mediante el código visto en el apartado anterior, fueron convolucionadas con la respuesta de impulso obtenida
en los recintos que escogimos.

<img src="images/foto_anecoica.jpeg" width="550" alt="voz_anecoica">

Los gráficos de la voz y el trompe sin ninguna modificación, además de los audios respectivos, son los siguientes:

<img src="images/voz_anecoica.png" width="550" alt="voz_anecoica">
<img src="images/trompe_anecoica.png" width="550" alt="trompe_anecoica">

a. Voz:

<audio controls src="../assets/audio/Grabaciones_propias/Voz_e_Instrumento/Grabacion de voz ANECOICA.wav" title="Title"></audio>

b. Trompe:

<audio controls src="../assets/audio/Grabaciones_propias/Voz_e_Instrumento/Grabacion de trompe ANECOICA.wav" title="Title"></audio>



## Recinto 1: Camara reverberante

En este recinto fue donde se realizaron las primeras grabaciones para obtener la respuesta de impulso.

A continuación, se aprecia un diagrama de posición para el recinto y una imagen del día de la medición:

<img src="images/foto_reverberante.jpeg" width="550" alt="voz_anecoica">
<img src="images/diagrama_posicion_reverberante.jpeg" width="217" alt="voz_anecoica">


Los gráficos de la voz y el trompe convolucionados con la señal de la cámara reverberante, además de los audios convolucionados respectivos, son los siguientes:

<img src="images/convolucion_voz_reverberante.png" width="550" alt="voz_anecoica">
<img src="images/convolucion_trompe_reverberante.png" width="550" alt="voz_anecoica">

a. Voz:

<audio controls src="../assets/audio/voz_convolucion_rev.wav" title="Title"></audio>

b. Trompe:

<audio controls src="../assets/audio/trompe_convolucion_rev.wav" title="Title"></audio>

## Recinto 2: Hall del edificio 9.000

En este recinto fue donde se realizaron las segundas grabaciones para obtener la respuesta de impulso.

A continuación, se aprecia un diagrama de posición para el recinto y una imagen del día de la medición:

<img src="images/foto_9k.jpeg" width="525" alt="voz_anecoica">
<img src="images/diagrama_posicion_9k.jpeg" width="566" alt="voz_anecoica">


Los gráficos de la voz y el trompe convolucionados con la señal del hall del edificio 9.000, además de los audios convolucionados respectivos, son los siguientes:

<img src="images/convolucion_voz_9k.png" width="550" alt="voz_anecoica">
<img src="images/convolucion_trompe_9k.png" width="550" alt="voz_anecoica">


a. Voz:

<audio controls src="../assets/audio/voz_convolucion_9k.wav" title="Title"></audio>

b. Trompe:

<audio controls src="../assets/audio/trompe_convolucion_9k.wav" title="Title"></audio>

## Recinto 3: Laboratorio Late 2

En este recinto fue donde se realizaron las ultimas grabaciones para obtener la respuesta de impulso.

A continuación, se aprecia un diagrama de posición para el recinto y una imagen del día de la medición:


<img src="images/foto_late2.jpeg" width="580" alt="voz_anecoica">
<img src="images/diagrama_posicion_late2.jpeg" width="441" alt="voz_anecoica">


Los gráficos de la voz y el trompe convolucionados con la señal del laboratorio LATE 2, además de los audios convolucionados respectivos, son los siguientes:


<img src="images/convolucion_voz_late2.png" width="550" alt="voz_anecoica">
<img src="images/convolucion_trompe_late2.png" width="550" alt="voz_anecoica">


a. Voz:

<audio controls src="../assets/audio/voz_convolucion_late.wav" title="Title"></audio>

b. Trompe:

<audio controls src="../assets/audio/trompe_convolucion_late.wav" title="Title"></audio>

# Conclusiones 

De acuerdo con las mediciones realizadas, se puede observar cómo cada sala modifica las señales sonoras según sus propias características acústicas. Mediante el proceso de convolución, fue posible recrear de forma fiel cómo un trabalenguas o un instrumento se percibirían como si estuvieran presentes en los distintos recintos medidos. Esto evidencia la importancia de estas herramientas en el procesamiento digital de señales, tanto para la emulación de espacios como para el diseño sonoro creativo.

# Bibliografia 

International Organization for Standardization. (2008). Acoustics—Measurement of room acoustic parameters—Part 2: Reverberation time in ordinary rooms (ISO 3382-2:2008). ISO.